In [112]:
import transformer
import torch
import torch.nn as nn
import importlib

importlib.reload(transformer)

<module 'transformer' from '/Users/desktop/Documents/a-slm/transformer.py'>

In [113]:
torch.manual_seed(42)

In [114]:
device = torch.device("mps")

vocab_size = 32

model = transformer.Transformer(
    num_l4g_blocks = 6,
    hidden_size = 48,
    intermediate_size = 128,
    num_q_heads = 6,
    num_kv_heads = 2,
    vocab_size = vocab_size,
    window_size = 3
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 1e-3
)

loss_fn = nn.CrossEntropyLoss()

In [115]:
input_ids = torch.randint(
    0,
    vocab_size,
    (2, 8),
    device=device
)

In [116]:
model.train()

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

for step in range(300):
    optimizer.zero_grad()

    logits = model(input_ids)

    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    loss = loss_fn(
        shift_logits.reshape(-1, vocab_size),
        shift_labels.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        print(
            f"step {step:3d} | loss {loss.item():.4f}"
        )

Total parameters:     742,224
Trainable parameters: 742,224
step   0 | loss 26.1697
step  25 | loss 0.0032
step  50 | loss 0.0005
step  75 | loss 0.0002
step 100 | loss 0.0002
step 125 | loss 0.0001
step 150 | loss 0.0001
step 175 | loss 0.0001
step 200 | loss 0.0001
step 225 | loss 0.0001
step 250 | loss 0.0001
step 275 | loss 0.0000
